# HuggingFace Entegrasyonu / HuggingFace Integration

Bu notebook, HuggingFace Hub'dan model yükleme ve kullanma örneklerini gösterir.
This notebook demonstrates how to load and use models from HuggingFace Hub.

## HuggingFace Space Farklılıkları / HuggingFace Space Differences:
HuggingFace Space uygulaması (huggingface_space/) farklı sınıf isimleri kullanır:
- **Defect classes**: `çizik`, `çatlak`, `delik`, `ezilme`, `yanık`, `pas`, `diğer`
- **Ore classes**: `manyetit`, `kromit`, `pirit`, `kalkopirit`, `atık`, `düşük tenörlü`

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path('../').resolve()
sys.path.insert(0, str(project_root))

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## HuggingFace Transformers Kütüphanesi / HuggingFace Transformers Library

In [ ]:
# Install required packages if needed
# !pip install transformers datasets accelerate

from transformers import AutoModelForImageClassification, AutoImageProcessor, pipeline
from transformers import AutoModelForObjectDetection, AutoProcessor

print("Transformers library imported successfully!")

## HuggingFace Hub'dan Model Yükleme / Loading Model from HuggingFace Hub

In [ ]:
# HuggingFace Space için Turkish class names (huggingface_space/app.py referansı)
# Defect classes
HF_DEFECT_CLASSES = [
    'çizik',      # Scratch
    'çatlak',     # Crack
    'delik',      # Hole
    'ezilme',     # Crushing
    'yanık',      # Burn
    'pas',        # Rust
    'diğer'       # Other
]

# Ore classes
HF_ORE_CLASSES = [
    'manyetit',       # Magnetite
    'kromit',         # Chromite
    'pirit',          # Pyrite
    'kalkopirit',     # Chalcopyrite
    'atık',           # Waste
    'düşük tenörlü'  # Low grade
]

print("HuggingFace Class Names:")
print(f"  Defect classes: {HF_DEFECT_CLASSES}")
print(f"  Ore classes: {HF_ORE_CLASSES}")

## HuggingFace Pipeline Kullanımı / Using HuggingFace Pipeline

In [ ]:
# Example: Using a pre-trained image classification model
# Not: Bu örnek genel bir model kullanır. Projeniz için özel model eğitmeniz gerekir.
# Note: This example uses a generic model. You need to train a custom model for your project.

# Create a simple image classification pipeline (using a sample model)
# Bu örnekteobilion/clip-vit-large-patch14 kullanıyoruz

def create_image_classifier_pipeline():
    """
    Create an image classification pipeline.
    
    Note: Replace with your custom trained model.
    """
    try:
        # Using a pre-trained CLIP model for demonstration
        # In production, use your fine-tuned model
        classifier = pipeline(
            "image-classification",
            model="openai/clip-vit-large-patch14",
            device=-1  # CPU (-1) or GPU (0)
        )
        return classifier
    except Exception as e:
        print(f"Error loading model: {e}")
        return None

print("Creating image classification pipeline...")
# classifier = create_image_classifier_pipeline()
# print("Pipeline created successfully!")

# Note: Commented out as it requires model download
print("\nNote: Uncomment the code above to use the pipeline.")
print("For production, use your custom trained model.")

## Özel Model Yükleme / Loading Custom Model

In [ ]:
# Custom model yükleme örneği / Example of loading custom model

def load_custom_model(model_path_or_hub_id, model_type="image-classification"):
    """
    Load a custom model from local path or HuggingFace Hub.
    
    Args:
        model_path_or_hub_id: Local path or HuggingFace model ID
        model_type: Type of model (image-classification, object-detection, etc.)
    
    Returns:
        Loaded model and processor
    """
    try:
        if os.path.exists(model_path_or_hub_id):
            # Load from local path
            print(f"Loading model from local path: {model_path_or_hub_id}")
            model = AutoModelForImageClassification.from_pretrained(model_path_or_hub_id)
            processor = AutoImageProcessor.from_pretrained(model_path_or_hub_id)
        else:
            # Load from HuggingFace Hub
            print(f"Loading model from HuggingFace Hub: {model_path_or_hub_id}")
            model = AutoModelForImageClassification.from_pretrained(model_path_or_hub_id)
            processor = AutoImageProcessor.from_pretrained(model_path_or_hub_id)
        
        return model, processor
    except Exception as e:
        print(f"Error loading model: {e}")
        return None, None

# Example usage (commented out as model doesn't exist)
# model, processor = load_custom_model("your-username/your-model")
print("Custom model loading function defined.")
print("\nExample usage:")
print('  model, processor = load_custom_model("your-username/your-defect-model")')

## Görüntü Sınıflandırma / Image Classification

In [ ]:
def classify_image_with_hf(image, classifier, class_labels):
    """
    Classify an image using HuggingFace pipeline.
    
    Args:
        image: PIL Image or numpy array
        classifier: HuggingFace pipeline
        class_labels: List of class labels
    
    Returns:
        Classification results
    """
    if classifier is None:
        return None
    
    # Convert numpy array to PIL if needed
    if isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    
    # Run classification
    results = classifier(image)
    
    return results

# Example: Create a mock classifier for demonstration
class MockClassifier:
    """Mock classifier for demonstration purposes."""
    
    def __init__(self, class_labels):
        self.class_labels = class_labels
    
    def __call__(self, image):
        """Return random predictions."""
        import random
        results = []
        for label in self.class_labels:
            score = random.random()
            results.append({'label': label, 'score': score})
        results = sorted(results, key=lambda x: x['score'], reverse=True)
        return results[:3]

# Create mock classifier with Turkish class labels
mock_classifier = MockClassifier(HF_ORE_CLASSES)
print("Mock classifier created with class labels:")
print(f"  {HF_ORE_CLASSES}")

## Örnek Görüntü ile Test / Testing with Sample Image

In [ ]:
# Create or load a sample image
SAMPLE_IMAGE_PATH = "../datasets/defect_detection/train/images/sample.jpg"

if os.path.exists(SAMPLE_IMAGE_PATH):
    image = cv2.imread(SAMPLE_IMAGE_PATH)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
else:
    # Create a test image
    print("Sample image not found. Creating a test image...")
    image_rgb = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)

print(f"Image shape: {image_rgb.shape}")

# Display the image
plt.figure(figsize=(6, 6))
plt.imshow(image_rgb)
plt.title("Test Görüntüsü / Test Image")
plt.axis('off')
plt.show()

# Run mock classification
results = mock_classifier(image_rgb)

print("\nClassification Results (Mock):")
print("-" * 40)
for result in results:
    print(f"  {result['label']}: {result['score']:.2%}")

## Yerel Model ile Karşılaştırma / Comparison with Local Model

In [ ]:
# Compare HuggingFace model with local YOLO model

def compare_models(image, yolo_model=None, hf_classifier=None):
    """
    Compare predictions from YOLO and HuggingFace models.
    
    Args:
        image: Input image
        yolo_model: YOLO model (optional)
        hf_classifier: HuggingFace classifier (optional)
    
    Returns:
        Comparison results
    """
    results = {
        'yolo': None,
        'huggingface': None
    }
    
    # YOLO inference
    if yolo_model is not None:
        yolo_results = yolo_model.predict(image, verbose=False)
        results['yolo'] = yolo_results
    
    # HuggingFace inference
    if hf_classifier is not None:
        hf_results = hf_classifier(image)
        results['huggingface'] = hf_results
    
    return results

# Example comparison (using mock data)
print("Model Comparison Function Defined")
print("\nUsage example:")
print("  from ultralytics import YOLO")
print("  yolo_model = YOLO('yolo11n-seg.pt')")
print("  results = compare_models(image, yolo_model, hf_classifier)")

## HuggingFace Hub'a Model Yükleme / Uploading Model to HuggingFace Hub

In [ ]:
# HuggingFace Hub'a model yükleme örneği / Example of uploading model to HuggingFace Hub

def upload_model_to_hub(model_path, repo_id, token=None):
    """
    Upload a trained model to HuggingFace Hub.
    
    Args:
        model_path: Local path to the model
        repo_id: HuggingFace repository ID (e.g., 'your-username/your-model')
        token: HuggingFace API token (optional, uses cached token if not provided)
    """
    try:
        from huggingface_hub import HfApi, create_repo
        
        # Create repository if it doesn't exist
        create_repo(repo_id, exist_ok=True)
        
        # Upload model files
        api = HfApi()
        api.upload_folder(
            folder_path=model_path,
            repo_id=repo_id,
            repo_type="model"
        )
        
        print(f"Model uploaded successfully to: https://huggingface.co/{repo_id}")
        return True
    except Exception as e:
        print(f"Error uploading model: {e}")
        return False

print("Model upload function defined.")
print("\nUsage:")
print('  upload_model_to_hub("./my_trained_model", "your-username/defect-detector")')
print("\nNote: You need to install huggingface_hub and authenticate first.")

## HuggingFace Space Dağıtımı / HuggingFace Space Deployment

In [ ]:
# HuggingFace Space yapılandırması / HuggingFace Space configuration

# Space için gerekli dosyalar / Required files for Space:
# - app.py: Gradio uygulaması / Gradio application
# - requirements.txt: Bağımlılıklar / Dependencies
# - hardware.yaml: Donanım yapılandırması / Hardware configuration

print("HuggingFace Space Deployment:")
print("=" * 50)
print("\n1. Required files in huggingface_space/:")
print("   - app.py: Main Gradio application")
print("   - requirements.txt: Python dependencies")
print("   - hardware.yaml: Hardware configuration")
print("\n2. Push to HuggingFace Hub:")
print("   - git clone https://huggingface.co/spaces/your-username/your-space")
print("   - Copy your files")
print("   - git add .")
print("   - git commit -m 'Update'")
print("   - git push")
print("\n3. Or use Spaces SDK:")
print("   - pip install huggingface_hub")
print("   - from huggingface_hub import create_space")
print("   - create_space(repo_id='your-username/space-name', sdk='gradio')")

## Turkish Class Names Karşılaştırması / Turkish Class Names Comparison

In [ ]:
# Compare class names between different modules

print("CLASS NAMES COMPARISON:")
print("=" * 50)

# From utils.py
print("\n1. utils.py (Streamlit App):")
print("   Ore classes: manyetit, krom, atık, düşük tenör, defekt, normal")
print("   Defect classes: çatlak, çizik, delik, leke, deformasyon")

# From huggingface_space/app.py
print("\n2. huggingface_space/app.py (Gradio):")
print(f"   Defect classes: {HF_DEFECT_CLASSES}")
print(f"   Ore classes: {HF_ORE_CLASSES}")

# From dataset.yaml
print("\n3. datasets/defect_detection/dataset.yaml (YOLO):")
print("   Classes: crack, scratch, dent, discoloration, contamination")

# Note
print("\n" + "=" * 50)
print("IMPORTANT: When deploying to HuggingFace Space, use HF classes!")
print("When using local YOLO model, use dataset.yaml classes.")

## Özet / Summary

Bu notebook şunları gösterdi:
- HuggingFace Transformers kütüphanesinin kullanımı
- HuggingFace Hub'dan model yükleme
- HuggingFace Pipeline kullanımı
- Yerel YOLO modeli ile karşılaştırma
- HuggingFace Hub'a model yükleme
- Farklı modüller arasındaki Türkçe sınıf isimlerinin karşılaştırması
---
This notebook demonstrated:
- Using HuggingFace Transformers library
- Loading models from HuggingFace Hub
- Using HuggingFace Pipeline
- Comparison with local YOLO model
- Uploading models to HuggingFace Hub
- Comparison of Turkish class names across different modules